In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from collections import defaultdict

In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
df = pd.read_csv('/content/drive/MyDrive/mushrooms.csv')

df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [34]:
df.replace('?', np.nan, inplace=True)

for col in df.columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [35]:
encoders = {}
for col in df.columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,1,5,2,4,1,6,1,0,1,4,...,2,7,7,0,2,1,4,2,3,5
1,0,5,2,9,1,0,1,0,0,4,...,2,7,7,0,2,1,4,3,2,1
2,0,0,2,8,1,3,1,0,0,5,...,2,7,7,0,2,1,4,3,2,3
3,1,5,3,8,1,6,1,0,1,5,...,2,7,7,0,2,1,4,2,3,5
4,0,5,2,3,0,5,1,1,0,4,...,2,7,7,0,2,1,0,3,0,1


In [36]:
X = df.drop('class', axis=1)
y = df['class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [37]:
class NaiveBayes:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.class_probs = {}
        self.feature_probs = {}

    def fit(self, X, y):
        n_samples = X.shape[0]
        self.classes = np.unique(y)

        # Prior P(y)
        for c in self.classes:
            self.class_probs[c] = np.sum(y == c) / n_samples


        self.feature_probs = {
            c: [defaultdict(float) for _ in range(X.shape[1])]
            for c in self.classes
        }

        for c in self.classes:
            X_c = X[y == c]

            for i in range(X.shape[1]):
                feature_values = X.iloc[:, i].unique()
                total_count = len(X_c)

                for val in feature_values:
                    count = np.sum(X_c.iloc[:, i] == val)

                    # Laplace smoothing
                    prob = (count + self.alpha) / (
                        total_count + self.alpha * len(feature_values)
                    )

                    self.feature_probs[c][i][val] = prob

    def predict(self, X):
        predictions = []

        for _, row in X.iterrows():
            class_scores = {}

            for c in self.classes:
                log_prob = np.log(self.class_probs[c])

                for i, val in enumerate(row):
                    prob = self.feature_probs[c][i].get(val, 1e-9)
                    log_prob += np.log(prob)

                class_scores[c] = log_prob

            predictions.append(max(class_scores, key=class_scores.get))


        return np.array(predictions)

In [38]:
model = NaiveBayes(alpha=1.0)

model.fit(X_train, y_train)

In [39]:
y_pred = model.predict(X_test)

In [40]:
accuracy = np.mean(y_pred == y_test)
print("Test Accuracy:", accuracy)

Test Accuracy: 0.9507692307692308


In [41]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Confusion Matrix:
 [[834   9]
 [ 71 711]]

Classification Report:

              precision    recall  f1-score   support

           0       0.92      0.99      0.95       843
           1       0.99      0.91      0.95       782

    accuracy                           0.95      1625
   macro avg       0.95      0.95      0.95      1625
weighted avg       0.95      0.95      0.95      1625



In [42]:
y_test_labels = encoders['class'].inverse_transform(y_test)
y_pred_labels = encoders['class'].inverse_transform(y_pred)

pd.DataFrame({
    "Actual": y_test_labels,
    "Predicted": y_pred_labels
}).head(20)

,Actual,Predicted
0,e,e
1,p,p
2,p,p
3,e,e
4,p,p
5,p,p
6,p,p
7,p,p
8,e,e
9,e,e
